# Lab 3: Adversarial Search

In this lab you will:

1. Implement minimax, and use it to solve tic-tac-toe.
2. See why minimax is not feasible in Connect 4, and fix it with a depth cutoff and an evaluation function.
3. Implement alpha-beta pruning, and measure how much work it saves.
4. Implement expectimax.

**What to submit:** commit this notebook **with its outputs**, plus `results_search.csv`, `results_matches.csv` and `predictions.json`.

<div style="border-left: 4px solid #0072B2; padding: 0.4em 1em; background: #f4f8fb;">

### How to use this notebook

* **Stub cells** hold a signature, a docstring, and `raise NotImplementedError`. Delete that line and write your code.
* **Check cells** run right after each stub. They name the position that broke, what should have come out, and what to look at. Add `verbose=True` for a full traceback.
* **Hints** are in collapsible blocks, ordered from a nudge to near-pseudocode.
* **Provided cells** are marked. You never write plotting, tournament, or export code.

</div>

## Part 0: The two games


In [ ]:
%run check_environment.py

In [ ]:
# PROVIDED
import math
import time

import numpy as np
import pandas as pd

from lab3kit.games import COLS, ROWS, Connect4, TicTacToe
from lab3kit.evaluation import windowed_eval, centre_first
from lab3kit.experiments import (MixedAgent, random_position, run_matches,
                                 run_search, SearchAgent)
from lab3kit.predictions import record_predictions, reveal
from lab3kit.results import save_results
from lab3kit import viz
from lab3kit.checks import (
    check_alphabeta_value,
    check_expectimax_value,
    check_minimax_cutoff,
    check_minimax_value,
)


def ready(*results):
    """Return True when the earlier cells this one depends on actually ran."""
    for result in results:
        if result is None:
            print("⏭  Skipped for now - run the experiment cell above first, "
                  "then come back.")
            return False
    return True


print("lab3kit loaded.")

### The interface the two games will use

So far you built agents that were free to act in a state space. We now consider a game, where there is another agent making moves with an opposing goal. We will use two games here.

Both games answer the same six questions:

| call | meaning |
|---|---|
| `game.initial_state()` | the starting position |
| `game.to_move(state)` | whose turn it is, 1 or 2 |
| `game.actions(state)` | the legal moves |
| `game.result(state, move)` | the position after a move |
| `game.is_terminal(state)` | whether the game is over |
| `game.utility(state, player)` | +1, 0 or −1, once it is over |

A state is a tuple `(board, player_to_move)`, and boards are tuples, so nothing can be changed in place. Each game also counts its own work in `game.nodes`.

In [ ]:
# PROVIDED
tic = TicTacToe()
tic_state = tic.initial_state()
for square in (4, 0, 8, 2): # play 4 moves
    tic_state = tic.result(tic_state, square)

four = Connect4()
four_state = four.initial_state()
for column in (3, 3, 4, 2, 4): # play 5 moves
    four_state = four.result(four_state, column)

figure, axes = viz.plt.subplots(1, 2, figsize=(9, 4))
viz.show_board(tic, tic_state, ax=axes[0])
viz.show_board(four, four_state, ax=axes[1])
figure.tight_layout()

Tic-tac-toe has no more than 9! = 362880 possible games. Connect 4 has about 4.5 trillion.

In this lab we will build a model that works on tic-tac-toe, but we will have to build a new one that works on Connect 4 due to this difference.

## Part 1: Predictions

Three guesses before you build anything. You are not graded on being right, only on going on the record first.

1. **Pruning.** Alpha-beta skips branches that cannot change the answer. Searching Connect 4 six plies deep, what fraction of minimax's positions do you think it will look at?
2. **Move ordering.** Will alpha-beta go faster if we look at middle column moves first, or slower? Write your prediction of how much faster middle columns would be compared to going in order.
3. **Opponent models.** Minimax assumes the opponent always plays their best move. Expectimax assumes they pick at random. Against an opponent that really does move completely at random, which one wins more games?

In [ ]:
predictions = record_predictions(
    alphabeta_node_fraction=None,   # a fraction between 0 and 1
    ordering_speedup=None,      # speed of looking at middle divided by speed of default
    random_opponent_winner=None,    # "minimax", "expectimax" or "tie"
    why="",                      # one or two sentences
)

## Part 2: Minimax

The idea in one sentence: **assume your opponent will make the move that is worst for you, and choose the move that is best given that.**

Your function returns the value of a position from a fixed player's point of view. So:

* At a node where it is *that* player's turn, take the **maximum** over the children.
  
* At a node where it is the *other* player's turn, take the **minimum**. From our agent's point of view, we assume the worst case.

The `player` argument is the point of view and stays fixed. `game.to_move(state)` is whose turn it is right now and alternates as you descend. Comparing the two is how you know which way to go.

The base minimax algorithm continues searching until it reaches the end of the game. When `game.is_terminal(state)` is true the answer is `game.utility(state, player)`, and there is nothing left to search.

### Task 1: `minimax_value`

**Your task:** Write `minimax_value(game, state, player)` returning the value of a position assuming best play from both sides, searching every line to the end.

In [ ]:
def minimax_value(game, state, player):
    """Return the value of this position, searching every line to the end.

    Args:
        game: The game being searched.
        state: The position to value.
        player: Whose point of view the value is measured from.

    Returns:
        +1 if player wins under best play, -1 if they lose, 0 for a draw.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_minimax_value(minimax_value)

<details><summary>Hint 1: where it stops</summary>

One `if` at the top: when `game.is_terminal(state)` is true, return `game.utility(state, player)`. Every branch of the recursion ends there.

</details>

<details><summary>Hint 2: the recursive step</summary>

Loop over `game.actions(state)`, build each `child = game.result(state, move)`, and call yourself on it with the **same** `player`. Collect the values.

</details>

<details><summary>Hint 3: which way to combine them</summary>

`if game.to_move(state) == player:` take the largest of the child values; otherwise take the smallest. Start your running best at `-math.inf` in the first case and `math.inf` in the second.

</details>

### Solving tic-tac-toe

*Provided code.* Now let's run your function on tic-tac-toe.

In [ ]:
# PROVIDED
try:
    tic.reset()
    started = time.time()
    value = minimax_value(tic, tic.initial_state(), 1)
    print(f"value of tic-tac-toe under best play: {value:+.0f}")
    print(f"{tic.nodes:,} positions searched in {time.time() - started:.2f} seconds")

    opening = []
    for square in tic.actions(tic.initial_state()):
        child = tic.result(tic.initial_state(), square)
        opening.append(minimax_value(tic, child, 1))
    print("value of each opening move:", [f"{v:+.0f}" for v in opening])
except NotImplementedError:
    print("⏭  Skipped for now - finish Task 1, then run this cell again.")

### Reflect
Can you guarantee a win with perfect tic-tac-toe play? What is the outcome of a perfect game?

## Part 3: Minimax in a more complex game


Now let's point the same function at Connect 4.

*Provided code.* We will run minimax over different starting points of the game, varying the number of pieces already on the board. Our code will measure the time it took to complete the search and the number of positions it had to look at.

In [ ]:
# PROVIDED
EXPLOSION = (34, 32, 30, 28)   # pieces already on the board

try:
    rows = []
    for pieces in EXPLOSION:
        board = Connect4()
        state = random_position(board, pieces, seed=1)
        empty = 0
        for cell in state[0]:
            if cell == 0:
                empty += 1
        board.reset()
        started = time.time()
        minimax_value(board, state, board.to_move(state))
        rows.append({"empty squares": empty,
                     "positions searched": board.nodes,
                     "seconds": round(time.time() - started, 2)})
    display(pd.DataFrame(rows))
except NotImplementedError:
    print("⏭  Skipped for now - finish Task 1, then run this cell again.")

Considering your results, can we run minimax on Connect-4 from the initial state? Connect 4 starts with 42 empty squares!

If we cannot search the entire tree start to finish, we will have to finish early. If the game didn't finish by the time we stop, we need a way to guess how well we are doing.

We call this **The evaluation function**. One is provided:

```python
game = Connect4(windowed_eval)
game.evaluate(state, player)     # positive means player is ahead
```

`windowed_eval` looks at every run of four squares in a line on the board. A run with just your squares give you points, and the more squares you have the more points. Tuns both players occupy are dead and score nothing. You gain some points for holding the centre column as well.

Note that this is much like the heuristic function from lab 2. It is a guess of how close we are to winning.

### Task 2: `minimax_cutoff`

Here you will build the same search, but it stops at a fixed depth and asks the evaluation function instead of running to the end.
his is the version that can actually play Connect 4.

Two changes to your Task 1 function:

* a `depth` argument, one lower in each recursive call;
* a second stopping condition: when `depth` reaches 0 on a position that is not over, return `game.evaluate(state, player)`.

Make sure to test `game.is_terminal(state)` *before* you test the depth, as this should override the evaluation function.

**Your task:** Write `minimax_cutoff(game, state, depth, player)`.

In [ ]:
def minimax_cutoff(game, state, depth, player):
    """Return the value of this position, stopping and guessing at depth 0.

    Args:
        game: The game being searched.
        state: The position to value.
        depth: How many more plies to look ahead.
        player: Whose point of view the value is measured from.

    Returns:
        A number. Larger is better for player.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_minimax_cutoff(minimax_cutoff, minimax_value)

<details><summary>Hint 1: start from Task 1</summary>

Copy `minimax_value` and add `depth` to the signature. Then pass `depth - 1` in every recursive call. Nothing else changes yet.

</details>

<details><summary>Hint 2: the second stopping condition</summary>

Under the terminal test, add `if depth == 0: return game.evaluate(state, player)`. Two stopping conditions, in that order.

</details>

<details><summary>Hint 3: a check you can run yourself</summary>

Give it more depth than tic-tac-toe can use (say 9 from an empty board) and it should return exactly what Task 1 returned. If it does not, the depth is interfering with a case it should not touch.

</details>

## Part 4: Alpha-beta pruning


Minimax looks at every leaf. However, a lot of times there is no point to looking at a leaf as there is no way it can change the outcome.

Suppose you are choosing between two moves. The first one, fully searched, guarantees you a value of **+3**. You start searching the second, and the opponent's very first reply already drops it to **+1**. You can stop: the opponent will pick *at least* that reply, so the second move is worth at most +1, which is already worse than the +3 you can have. Whatever else is down that branch, it cannot change your decision.

Alpha-beta uses this to *prune* the search tree. It keeps track of two values:

* **alpha** - the best value the maximising side has secured anywhere so far
* **beta** - the best value the minimising side has secured anywhere so far

At a maximising node, when your running value reaches `beta` or more, alpha-beta will stop and return. At a minimising node, when your running value drops to `alpha` or below, stop and return.

The result is exactly the value `minimax_cutoff` would have returned (not an approximation!)

### Move ordering

The pruning only fires once you have found something good to compare against, so the earlier a good move is tried, the more gets cut. `game.order(state, moves)` returns the moves in the order the game suggests, which for Connect 4 means middle columns first. Use it instead of `game.actions(state)` alone and the experiment below will show what it is worth.

### Task 3: `alphabeta_value`

**Your task:** Write `alphabeta_value(game, state, depth, player, alpha, beta)`. It must return exactly what `minimax_cutoff` returns, and it must search fewer positions.

Start from your Task 2 solution and use the stopping conditions.

In [ ]:
def alphabeta_value(game, state, depth, player, alpha=-math.inf, beta=math.inf):
    """Return the same value as minimax_cutoff, skipping branches that cannot matter.

    Args:
        game: The game being searched.
        state: The position to value.
        depth: How many more plies to look ahead.
        player: Whose point of view the value is measured from.
        alpha: Best value the maximising side has secured so far.
        beta: Best value the minimising side has secured so far.

    Returns:
        A number, equal to what minimax_cutoff would return.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_alphabeta_value(alphabeta_value, minimax_cutoff)

<details><summary>Hint 1: start from Task 2</summary>

Copy `minimax_cutoff` and add `alpha` and `beta` to the signature with their defaults. Pass them down every recursive call. Nothing else changes yet.

</details>

<details><summary>Hint 2: the cutoff at a maximising node</summary>

Right after updating your running `value`, check `if value >= beta:` and return immediately. Then update `alpha` to `max(alpha, value)` before moving on to the next move. Order matters: test first, then widen alpha.

</details>

<details><summary>Hint 3: the minimising node is the mirror image</summary>

Check `if value <= alpha:` and return; then narrow `beta` to `min(beta, value)`. If you find yourself updating beta at the maximising node, the two halves have been swapped and your values will come out wrong.

</details>

<details><summary>Hint 4: getting the ordering in</summary>

Replace `game.actions(state)` with `game.order(state, game.actions(state))`. When a game has no ordering set, that hands the moves straight back, so it is always safe.

</details>

### Experiment 1: how much does pruning save?

*Provided code.* The same position searched to each depth three ways: your Task 2 cutoff search, alpha-beta with moves in column order, and alpha-beta with the middle columns first. The cutoff version stops at depth 6 because depth 7 would take too long.

In [ ]:
# PROVIDED
SEARCHES = (
    ("minimax", None),
    ("alpha-beta", None),
    ("alpha-beta + ordering", centre_first),
)

df_search = None
try:
    rows = []
    for depth in (2, 3, 4, 5, 6, 7, 8):
        for label, order in SEARCHES:
            if label == "minimax":
                if depth > 6:
                    continue
                value_fn = minimax_cutoff
            else:
                value_fn = alphabeta_value
            game = Connect4(windowed_eval, order_fn=order)
            row = run_search(game, game.initial_state(), depth, value_fn)
            row["search"] = label
            rows.append(row)
    df_search = pd.DataFrame(rows)
except NotImplementedError:
    print("⏭  Skipped for now - finish Tasks 2 and 3, then run this cell again.")

In [ ]:
# PROVIDED
if ready(df_search):
    viz.plot_nodes_by_depth(df_search)
    viz.plot_branching(df_search, reference={"b": 7.0,"b^0.75": 7 ** 0.75,"b^0.5": 7 ** 0.5})
    table = df_search.pivot_table(index="depth", columns="search", values="nodes")
    display(table.astype("Int64"))

If a search looks at $N$ positions over $d$ plies, then $N^{1/d}$ is the effective branching factor: the average number of moves it really had to consider at each step. Connect 4 has seven columns, so plain minimax should sit at about 7.

Theory says alpha-beta gets that down to about $b^{3/4}$ with moves in an arbitrary order, and to about $b^{1/2}$ with the best possible ordering. Those are the dashed lines.

Take a moment to think about your results and the relation to the theoretical lines.

## Part 5: A different opponent model

Minimax assumes the opponent plays their best move, every time. That is the safe assumption, and against a strong opponent it is the right one.

But what if the opponent is picking moves at random? Perhaps minimax would be too cautious, expecting the opponent to play perfectly even when they don't.

**Expectimax** takes the opposite view. At the opponent's nodes it does not take the minimum; it takes the **average** of the children, which is what you would expect if every legal move were equally likely. Your own nodes are unchanged and you still choose your best move.

That is the entire difference: `min(...)` becomes `sum(...) / len(...)`.

### Task 4: `expectimax_value`

**Your task.** Write `expectimax_value(game, state, depth, player)`. Start from your Task 2 function and change the opponent's nodes.

Note that there is no alpha-beta version of this. Pruning works because a minimising node can only ever go down, so a bound can rule a branch out. An average can be pulled back up by a later child, so nothing can be ruled out early.

In [ ]:
def expectimax_value(game, state, depth, player):
    """Return the value of this position, assuming the opponent moves at random.

    Args:
        game: The game being searched.
        state: The position to value.
        depth: How many more plies to look ahead.
        player: Whose point of view the value is measured from.

    Returns:
        A number. Larger is better for player.
    """
    raise NotImplementedError  # your code here

In [ ]:
check_expectimax_value(expectimax_value, minimax_cutoff)

<details><summary>Hint 1: what stays the same</summary>

Both stopping conditions, and the whole maximising branch. Copy your Task 2 function and leave those alone.

</details>

<details><summary>Hint 2: the opponent's node</summary>

Instead of tracking a running minimum, keep a running total. Add each child's value to it, and when the loop is done divide by `len(moves)`.

</details>

<details><summary>Hint 3: do not forget to divide</summary>

Returning the sum instead of the average makes every value grow with the number of legal moves, so positions with more options look better regardless of whether they are. The check cell tests for this.

</details>

## Part 6: Which assumption costs less?

### Experiment 2: the tournament

*Provided code.* Both agents search four plies. They play an opponent that mixes randomness with a two-ply search, controlled by `epsilon`: at `epsilon = 0` the opponent always takes its best two-ply move, at `epsilon = 1` it plays completely at random, and in between it flips a weighted coin each turn.

Twenty games at each setting, each agent playing first in half of them. This may take 30 seconds or so.

In [ ]:
# PROVIDED
def make_connect4(seed):
    """A fresh Connect 4 game, with the provided evaluation and move ordering."""
    return Connect4(windowed_eval, order_fn=centre_first)


TOURNAMENT_SEEDS = list(range(20))
AGENT_DEPTH = 4
EPSILONS = (0.0, 0.25, 0.5, 0.75, 1.0)

df_matches = None
try:
    baseline = SearchAgent(alphabeta_value, 2, "two-ply")
    frames = []
    for epsilon in EPSILONS:
        agents = {
            "minimax": SearchAgent(alphabeta_value, AGENT_DEPTH, "minimax"),
            "expectimax": SearchAgent(expectimax_value, AGENT_DEPTH, "expectimax"),
        }
        frame = run_matches(agents, MixedAgent(baseline, epsilon),
                            make_connect4, TOURNAMENT_SEEDS)
        frame["epsilon"] = epsilon
        frames.append(frame)
    df_matches = pd.concat(frames, ignore_index=True)
except NotImplementedError:
    print("⏭  Skipped for now - finish Tasks 3 and 4, then run this cell again.")

In [ ]:
# PROVIDED
if ready(df_matches):
    viz.plot_opponent_sweep(df_matches)
    summary = df_matches.groupby(["epsilon", "agent"]).agg(
        score=("result", "mean"),
        plies=("plies", "mean"),
    )
    display(summary)

Both panels show the same games. The left one counts who won; the right one counts how long the won games took. A missing point on the right means that agent won nothing at all at that setting, so there is no data.

In [ ]:
# PROVIDED
if ready(df_search, df_matches):
    save_results(df_search, df_matches)

In [ ]:
# PROVIDED
if ready(df_search, df_matches):
    display(reveal(predictions, df_search, df_matches))

### Reflect

Answer each question in 3 to 6 sentences, in the markdown cell that follows it.

**1.** Alpha-beta returns exactly the same value as minimax while searching a small fraction of the positions. Explain how it can skip work without ever changing the answer, and why trying the middle columns first makes it skip more.

*Your answer here.*

**2.** Look at the two panels of the tournament figure. Against a fully random opponent, which agent would you call better, and does your answer depend on which panel you look at? What does that tell you about reporting a single number for "how good is this agent"?

*Your answer here.*

**3.** Expectimax models the opponent more accurately when the opponent really is random. Given that, why does it not win more games, and why is it so much worse at `epsilon = 0`? What is the safer assumption to make if you don't know your opponent?

*Your answer here.*

**4.** When is an evaluation function necessary? Why can't we always just use the utility function?

*Your answer here.*

## Where this is going

How do we build a better model of how an opponent will behave than our expectimax? This is something we will think about going forwrad.

If time permits, we will also look at MCTS.

---

**Before you submit:** run this notebook top to bottom one last time (Kernel → Restart Kernel and Run All Cells), check that `results_search.csv`, `results_matches.csv` and `predictions.json` are in the folder, and commit all four files.